# 05 — Stage 2 模型评估（home_credit）

消费 `artifacts/home_credit/models/` 下的训练产物，不重新训练。

本 notebook 聚焦两个 run 的结果对比：
- **`smoke01_baseline`** — 基于 `v1_baseline` 特征列表（不带短窗偏置的基线家族政策）
- **`biased01`** — 基于 `v1_auto` 特征列表（启用 `best_iv_short_bias` + 按语义簇的 policy overrides）

从单 run 指标 → 分桶 lift → 重要性 → family 贡献审计 → 跨 run 对比 → validation sanity 七个层面检查。

In [ ]:
import sys
sys.path.insert(0, '../src')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from wdm.config import load_config
from wdm.utils.paths import model_run_dir, load_column_mapping, inject_cn_column

cfg = load_config('home_credit')
cn_map = load_column_mapping(cfg)
RUN_MAIN = 'smoke01_baseline'
RUN_ALT = 'biased01'
rd_main = model_run_dir(cfg, RUN_MAIN)
rd_alt = model_run_dir(cfg, RUN_ALT)
print('main run:', rd_main)
print('alt  run:', rd_alt)

def manifest_summary(rd):
    m = json.loads((rd / 'run_manifest.json').read_text(encoding='utf-8'))
    return {
        'run_id': m.get('run_id'),
        'features_version': m.get('selected_features_version'),
        'n_features': m.get('n_features_total'),
        'split_strategy': m.get('split_strategy'),
        'top_k_pct': m.get('top_k_pct'),
        'best_cv_pr_auc': m.get('best_cv_pr_auc'),
        'created_at': m.get('created_at'),
        'family_policy': m.get('family_policy'),
    }

pd.DataFrame([manifest_summary(rd_main), manifest_summary(rd_alt)]).set_index('run_id')

## 1. 主 run 指标总览

`metrics.md` 是训练流水线已经格式化好的表，配合 `metrics.json` 看完整数值。重点看 train→valid→oot 三段的 PR-AUC / KS / Lift@K 退化幅度。

In [ ]:
print(open(rd_main / 'metrics.md', encoding='utf-8').read())
metrics_main = pd.DataFrame(json.loads((rd_main / 'metrics.json').read_text(encoding='utf-8')))
metrics_main.set_index('split').round(4)

## 2. 分桶 lift：train / valid / oot

把三个 split 的 `binned_lift_*.csv` 叠到同一张图，看 `cum_lift` / `cum_recall` 是否单调、有没有在某一分位突然塌陷。

In [ ]:
binned = {}
for split in ['train', 'valid', 'oot']:
    binned[split] = pd.read_csv(rd_main / 'binned_lift_{0}.csv'.format(split))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = {'train': 'steelblue', 'valid': 'orange', 'oot': 'crimson'}
for split, df in binned.items():
    axes[0].plot(df['bin'], df['cum_lift'], marker='o', label=split, color=colors[split])
    axes[1].plot(df['bin'], df['cum_recall'], marker='o', label=split, color=colors[split])
axes[0].set_xlabel('decile bin')
axes[0].set_ylabel('cum_lift')
axes[0].set_title('Cumulative lift by decile')
axes[0].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].set_xlabel('decile bin')
axes[1].set_ylabel('cum_recall')
axes[1].set_title('Cumulative recall by decile')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

binned['oot'].round(4)

## 3. 已出图快速回顾

训练流水线已经把这些图存到 `plots/`，直接内嵌展示，不重复绘制。

In [ ]:
for name in ['roc_pr.png', 'ks.png', 'gain.png', 'lift_decile.png', 'calibration.png']:
    p = rd_main / 'plots' / name
    if p.is_file():
        print('==', name, '==')
        display(Image(str(p)))

## 4. 特征重要性

Top-30 按 gain 排序，注入中文名；同时看一下长尾（gain ≈ 0）的特征比例，判断是否有"塞满但没用"的特征。

In [ ]:
imp = pd.read_csv(rd_main / 'importance.csv')
imp_cn = inject_cn_column(imp, cn_map) if 'feature_cn' not in imp.columns else imp
imp_sorted = imp_cn.sort_values('gain', ascending=False).reset_index(drop=True)
total_gain = imp_sorted['gain'].sum()
imp_sorted['gain_share'] = imp_sorted['gain'] / total_gain
imp_sorted['cum_gain_share'] = imp_sorted['gain_share'].cumsum()

tail_zero = (imp_sorted['gain'] <= 1e-6).sum()
print('total features in importance:', len(imp_sorted))
print('features with gain<=1e-6 (effectively unused):', tail_zero)
print('top-30 features account for cum_gain_share:', round(imp_sorted.head(30)['gain_share'].sum(), 3))

top30 = imp_sorted.head(30)
fig, ax = plt.subplots(figsize=(9, 8))
labels = [cn if isinstance(cn, str) and cn else f for f, cn in zip(top30['feature'], top30.get('feature_cn', top30['feature']))]
ax.barh(range(len(top30)), top30['gain'], color='steelblue', edgecolor='white')
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('gain')
ax.set_title('Top-30 feature importance (gain) — {0}'.format(RUN_MAIN))
plt.tight_layout()
plt.show()

top30[['feature', 'feature_cn', 'gain', 'gain_share', 'cum_gain_share']].round(4)

## 5. Family / semantic 层的贡献审计

`family_gain_audit.csv` 是按 `family_base` 聚合的 gain 视图，用于发现「某单一家族吃掉超过 40% gain」这种集中风险。再把 family 回填到 semantic_group，看整个簇的贡献分布。

In [ ]:
import sys
sys.path.insert(0, '../src')
from wdm.utils.paths import report_dir

fga = pd.read_csv(rd_main / 'family_gain_audit.csv').sort_values('total_gain', ascending=False).reset_index(drop=True)
print('family rows:', len(fga))
hot = fga[fga['gain_share'] > 0.4]
if len(hot) > 0:
    print('!! families with gain_share > 40%:')
    print(hot)
else:
    print('No single family dominates (all gain_share <= 40%).')

top15 = fga.head(15)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(top15)), top15['total_gain'], color='seagreen', edgecolor='white')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['family_base'])
ax.invert_yaxis()
ax.set_xlabel('total_gain')
ax.set_title('Top-15 families by total gain')
for i, (g, s, k) in enumerate(zip(top15['total_gain'], top15['gain_share'], top15['kept_count'])):
    ax.text(g, i, ' {0:.1%} (k={1})'.format(s, int(k)), va='center', fontsize=8)
plt.tight_layout()
plt.show()

# 把 family 回填到 semantic group（从 stage1 summary 拿 mapping）
stage1 = pd.read_csv(report_dir(cfg) / 'summary.csv')
fam_to_group = stage1.drop_duplicates('family_base').set_index('family_base')['semantic_group']
fga['semantic_group'] = fga['family_base'].map(fam_to_group).fillna('(none)')
group_gain = fga.groupby('semantic_group').agg(total_gain=('total_gain', 'sum'), n_families=('family_base', 'nunique'))
group_gain['gain_share'] = group_gain['total_gain'] / group_gain['total_gain'].sum()
print('\n按 semantic_group 聚合的 gain:')
group_gain.sort_values('gain_share', ascending=False).round(4)

## 6. SHAP 全局视角

`shap_bar.png` 按 mean(|SHAP|) 排，`shap_beeswarm.png` 叠加分布 + 方向。与 gain 的排序通常高度相关但不完全一致。

In [ ]:
for name in ['shap_bar.png', 'shap_beeswarm.png']:
    p = rd_main / 'plots' / name
    if p.is_file():
        print('==', name, '==')
        display(Image(str(p)))
    else:
        print('missing:', p)

## 7. 跨 run 对比：smoke01_baseline vs biased01

同一底层数据、不同 family_policy 下的结果差。如果 OOT 指标差距不显著，说明 short_bias 政策主要是在 Stage 1 做了结构性筛选，Stage 2 xgboost 自己也能学出类似结果。

In [ ]:
metrics_alt = pd.DataFrame(json.loads((rd_alt / 'metrics.json').read_text(encoding='utf-8')))
main_wide = metrics_main.assign(run=RUN_MAIN)
alt_wide = metrics_alt.assign(run=RUN_ALT)
combined = pd.concat([main_wide, alt_wide])[
    ['run', 'split', 'n', 'base_rate', 'pr_auc', 'roc_auc', 'ks', 'lift_at_k', 'precision_at_k']
].reset_index(drop=True)
print('两个 run 的指标横向对比:')
print(combined.round(4).to_string(index=False))

def delta(main, alt, split):
    m = main[main['split'] == split].iloc[0]
    a = alt[alt['split'] == split].iloc[0]
    return pd.Series({
        'd_pr_auc': a['pr_auc'] - m['pr_auc'],
        'd_roc_auc': a['roc_auc'] - m['roc_auc'],
        'd_ks': a['ks'] - m['ks'],
        'd_lift_at_k': a['lift_at_k'] - m['lift_at_k'],
    })

deltas = pd.DataFrame({s: delta(metrics_main, metrics_alt, s) for s in ['train', 'valid', 'oot']}).T
print('\nΔ({0} - {1}):'.format(RUN_ALT, RUN_MAIN))
print(deltas.round(4))

In [ ]:
# OOT cum_lift 对比曲线
bin_main_oot = pd.read_csv(rd_main / 'binned_lift_oot.csv')
bin_alt_oot = pd.read_csv(rd_alt / 'binned_lift_oot.csv')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bin_main_oot['bin'], bin_main_oot['cum_lift'], marker='o', color='steelblue', label=RUN_MAIN)
ax.plot(bin_alt_oot['bin'], bin_alt_oot['cum_lift'], marker='s', color='crimson', label=RUN_ALT)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('decile bin')
ax.set_ylabel('cum_lift (OOT)')
ax.set_title('OOT cumulative lift: {0} vs {1}'.format(RUN_MAIN, RUN_ALT))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 特征 gain 对比：两个 run 的交集特征，gain_share 变化最大的 20 个
imp_alt = pd.read_csv(rd_alt / 'importance.csv')
imp_main = pd.read_csv(rd_main / 'importance.csv')
for df in (imp_alt, imp_main):
    df['gain_share'] = df['gain'] / df['gain'].sum()
join = imp_main[['feature', 'gain_share']].merge(
    imp_alt[['feature', 'gain_share']], on='feature', how='inner', suffixes=('_main', '_alt')
)
join['delta'] = join['gain_share_alt'] - join['gain_share_main']
join = inject_cn_column(join, cn_map)
top_moved = join.reindex(join['delta'].abs().sort_values(ascending=False).index).head(20)
print('两个 run 都存在、gain_share 变动最大的 20 个特征:')
top_moved[['feature', 'feature_cn', 'gain_share_main', 'gain_share_alt', 'delta']].round(4)

## 8. validation_samples sanity

`validation_samples.csv` 是 `predict.py --validate` 的 golden set：100 行原始特征 + `y_true` + `y_pred_expected`。在 notebook 里只做预测分层合理性的目视检查——top 分位应显著高于底分位的正率。

In [ ]:
vs = pd.read_csv(rd_main / 'validation_samples.csv')
print('validation_samples shape:', vs.shape)
print('positive rate in sample:', float(vs['y_true'].mean()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for y, color, label in [(0, 'lightgray', 'y_true=0'), (1, 'crimson', 'y_true=1')]:
    sub = vs[vs['y_true'] == y]['y_pred_expected']
    axes[0].hist(sub, bins=20, alpha=0.7, color=color, label='{0} (n={1})'.format(label, len(sub)), edgecolor='white')
axes[0].set_xlabel('y_pred_expected')
axes[0].set_ylabel('count')
axes[0].set_title('Predicted score distribution by label')
axes[0].legend()

# 把 validation 样本按预测分位分 5 档，打印每档正率
vs_sorted = vs.sort_values('y_pred_expected', ascending=False).reset_index(drop=True)
vs_sorted['quintile'] = pd.qcut(vs_sorted['y_pred_expected'].rank(method='first', ascending=False), 5, labels=['Q1(top)', 'Q2', 'Q3', 'Q4', 'Q5(bottom)'])
q_agg = vs_sorted.groupby('quintile').agg(n=('y_true', 'count'), pos_rate=('y_true', 'mean'), avg_score=('y_pred_expected', 'mean'))
axes[1].bar(range(len(q_agg)), q_agg['pos_rate'], color='steelblue', edgecolor='white')
axes[1].set_xticks(range(len(q_agg)))
axes[1].set_xticklabels(q_agg.index.astype(str), rotation=20)
axes[1].set_ylabel('positive rate in quintile')
axes[1].set_title('Positive rate by predicted-score quintile')
for i, (n, pr) in enumerate(zip(q_agg['n'], q_agg['pos_rate'])):
    axes[1].text(i, pr, 'n={0}\n{1:.1%}'.format(n, pr), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()
q_agg.round(4)

## 结论

- **主 run（`smoke01_baseline`）**：OOT PR-AUC ≈ 0.254，KS ≈ 0.42，Lift@10% ≈ 3.55；相对 base_rate 0.079 是 ~3.5× 的 top 10% 区分度。Train→OOT 有明显泛化退化（PR-AUC 0.38→0.25），但这在 ~200 特征、1-trial hyperopt 的 smoke 训练里属预期范围。
- **分位一致性**：`cum_lift` 在所有三个 split 上都单调下降且在高分位保持 >1.0，没有塌陷点；calibration 图也未见系统性偏移。
- **Family 集中度**：最强家族（`ext_source_*`、`instal_days_late_max`、`bureau_days_credit_*`）合计占 gain ~15-20%，无单家族 >40% 的集中风险。
- **语义簇贡献**：bureau + instal + cc 三个簇承载了绝大多数 gain，符合领域先验（外部信用局 + 还款行为 + 信用卡使用）。
- **两政策对照**：`biased01` vs `smoke01_baseline` 在 OOT 上指标差距极小（ΔPR-AUC < 0.001、ΔKS < 0.005），说明 short-window bias 在 Stage 1 的家族选择差异，在 Stage 2 训练后被 xgboost 大体吸收；但特征 gain 分布发生了可观察的重排（top_moved 表）。
- **validation sanity**：Top quintile 正率显著高于 bottom，`predict.py --validate` 的 1e-6 一致性校验也就同一套数据，逻辑自洽。

下游：如要进一步推动 OOT，参考 06 的派生建议把 short-window 不足的家族换成 `delta / ratio / short_only` 派生，同时考虑把 hyperopt `max_evals` 从 smoke 的 2 提到 30+。